This is a Notebook file where we evaluate the models using different graphs, metrics and visualisations. 

In [ ]:
import pandas as pd
import numpy as np

import pickle
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    roc_curve,
    auc,
)

path_png = "D:/Users/nickt/ECO482/Research Project Files/Visualizations/pngs"

Load in the model results from models.py

In [336]:
model_path = "D:/Users/nickt/ECO482/Research Project Files/Models/trained_models.pkl"

with open(model_path, "rb") as f:
    saved_data = pickle.load(f)

best_models = saved_data["best_models"]
summary_df = saved_data["summary_df"]
evaluation_results = saved_data["evaluation_results"]
X = saved_data["X"]
y = saved_data["y"]
feature_names = saved_data["feature_names"]

summary_df.head()

,model,best_params,best_cv_recall
0,XGBoost,"{'colsample_bytree': 0.7, 'learning_rate': 0.1...",0.014286
1,Random Forest,"{'class_weight': 'balanced', 'max_depth': 4, '...",0.229524


Summary Statistics for Each Model

In [ ]:
results_df = pd.DataFrame(evaluation_results)
summary_table = results_df.groupby('model')[['accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].mean()
summary_std = results_df.groupby('model')[['accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].std()

summary_display = results_df[['model', 'fold', 'accuracy', 'recall', 'precision', 'roc_auc', 'avg_precision']].copy()
summary_display = summary_display.round(4)

print("\n" + "="*40)
print("MODEL EVALUATION SUMMARY")
print("="*40)
print(summary_display.to_string(index=False))
print("\nAVERAGE PERFORMANCE BY MODEL:")
print(summary_table.round(4))
print("\nSTANDARD DEVIATION BY MODEL:")
print(summary_std.round(4))


SAVED MODEL EVALUATION SUMMARY
        model  fold  accuracy  recall  precision  roc_auc  avg_precision
      XGBoost     1    0.8738  0.0000     0.0000      NaN         0.0000
      XGBoost     2    1.0000  0.0000     0.0000      NaN         0.0000
      XGBoost     3    0.8447  0.0714     0.2500   0.7440         0.4123
      XGBoost     4    1.0000  0.0000     0.0000      NaN         0.0000
      XGBoost     5    0.7087  0.0000     0.0000   0.9539         0.9372
Random Forest     1    0.5340  0.0000     0.0000      NaN         0.0000
Random Forest     2    0.9417  0.0000     0.0000      NaN         0.0000
Random Forest     3    0.8350  0.7143     0.4348   0.7953         0.3851
Random Forest     4    0.9903  0.0000     0.0000      NaN         0.0000
Random Forest     5    0.8252  0.4333     0.9286   0.9315         0.8855

AVERAGE PERFORMANCE BY MODEL:
               accuracy  recall  precision  roc_auc  avg_precision
model                                                             


Below we create visualizations based on each models performance. We first create and compare ROC Curves for each Model

In [ ]:
roc_fig = go.Figure()

for model_name in best_models.keys():
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_score = np.concatenate([r['y_proba'] for r in model_results])

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    roc_fig.add_trace(go.Scatter(
        x=fpr, 
        y=tpr, 
        mode='lines', 
        name=f'{model_name} (AUC={roc_auc:.3f})')
    )

roc_fig.add_trace(go.Scatter(
    x=[0, 1], 
    y=[0, 1], 
    mode='lines', 
    name='Random Chance', 
    line=dict(dash='dash', color='gray'))
    )

roc_fig.update_layout(
    title={'text':'ROC Curves', 'x':0.5, 'xanchor':'center'}, 
    xaxis_title='False Positive Rate', 
    yaxis_title='True Positive Rate', 
    width=900,
    height=500, 
    template='simple_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="right",
        x=0.8
    )
)

roc_fig.show()
# roc_fig.write_image(f"{path_png}/roc_curves.png", scale=2)

Precision Recall Curves

In [ ]:
pr_fig = go.Figure()

for model_name in best_models.keys():
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_score = np.concatenate([r['y_proba'] for r in model_results])

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    avg_prec = average_precision_score(y_true, y_score)
    pr_fig.add_trace(go.Scatter(
        x=recall, 
        y=precision, 
        mode='lines', 
        name=f'{model_name} (AP={avg_prec:.3f})')
    )

pr_fig.update_layout(
    title={'text':'Precision-Recall Curves', 'x':0.5, 'xanchor':'center'}, 
    xaxis_title='Recall', 
    yaxis_title='Precision', 
    width=900,
    height=500, 
    template='simple_white',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="right",
        x=0.75
    )
)
pr_fig.show()
# pr_fig.write_image(f"{path_png}/pr_curves.png", scale=2)

In [ ]:
cm_list = []

for model_name in best_models.keys():
    model_results = [r for r in evaluation_results if r['model'] == model_name]
    y_true = np.concatenate([r['y_true'] for r in model_results])
    y_pred = np.concatenate([r['y_pred'] for r in model_results])
    cm = confusion_matrix(y_true, y_pred)
    cm_list.append((model_name, cm))

shared_max = max(cm.max() for _, cm in cm_list)

for model_name, cm in cm_list:
    fig = go.Figure(
        data=go.Heatmap(
            z=cm,
            x=['No Inversion', 'Inversion'],
            y=['No Inversion', 'Inversion'],
            text=cm,
            texttemplate='%{text}',
            colorscale='Cividis',
            zmin=0,
            zmax=shared_max,
            showscale=True,
        )
    )

    fig.update_layout(
        title={'text':f'{model_name} Confusion Matrix', 'x':0.5, 'xanchor':'center'},
        xaxis_title='Predicted label',
        yaxis_title='True label',
        width=700,
        height=500,
        template='plotly_white'
    )

    fig.show()
    # fig.write_image(f"{path_png}/{model_name}_confusion_matrix.png", scale=2)

Feature Importance

In [ ]:
# XGBoost feature importance
xgb_importances = best_models['XGBoost'].feature_importances_
xgb_df = pd.DataFrame({'Feature': feature_names, 'Importance': xgb_importances}).sort_values(by='Importance', ascending=False)
xgb_top = xgb_df.head(15)[::-1]

fig_xgb = px.bar(
    xgb_top,
    x='Importance',
    y='Feature',
    orientation='h',
    title='XGBoost Feature Importance',
    labels={'Importance': 'Importance', 'Feature': 'Feature'}
)
fig_xgb.update_traces(
    marker_color='#4C78A8',
    marker_line_color='black',
    marker_line_width=1, 
    marker_pattern_shape="/"
)

fig_xgb.update_layout(
    title={'x':0.5, 'xanchor':'center'},
    yaxis={'categoryorder':'total ascending'}, 
    width=900,
    height=700, 
    template='plotly_white',
)
fig_xgb.show()
# fig_xgb.write_image(f"{path_png}/xgb_feature_importance.png", scale=2)

# Random forest feature importance
rf_importances = best_models['Random Forest'].feature_importances_
rf_df = pd.DataFrame({'Feature': feature_names, 'Importance': rf_importances}).sort_values(by='Importance', ascending=False)
rf_top = rf_df.head(15)[::-1]

fig_rf = px.bar(
    rf_top,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Random Forest Feature Importance',
    labels={'Importance': 'Importance', 'Feature': 'Feature'}
)
fig_rf.update_traces(
    marker_color='#F58518',
    marker_line_color='black',
    marker_line_width=1, 
    marker_pattern_shape="/"
)
fig_rf.update_layout(
    title={'x':0.5, 'xanchor':'center'},
    yaxis={'categoryorder':'total ascending'}, 
    width=900,
    height=700, 
    template='plotly_white',
)
fig_rf.show()
# fig_rf.write_image(f"{path_png}/rf_feature_importance.png", scale=2)